# Collect cs.AI metadata, January 2022–October 2026

Run the offline tests first from the repository root:

```bash
python3 -m unittest discover -s tests -v
```

Then restart the kernel and run this notebook yourself. No collection has been executed by the assistant.

**Scope:** papers first submitted in this period with cs.AI as a primary or additional category. Application papers remain included. These records do not represent all AI research. Returned titles, abstracts, and revision dates reflect current metadata, not necessarily the version available at the submission date. No PDFs or full texts are downloaded.

This notebook collects and validates data; cleaning and research analysis come later. [arXiv API documentation](https://info.arxiv.org/help/api/user-manual.html).


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import sys

import pandas as pd
from IPython.display import display

# Support Jupyter started from the repository root or notebooks/.
project_root = Path.cwd()
if not (project_root / "arxiv_collection.py").is_file():
    project_root = project_root.parent
if not (project_root / "arxiv_collection.py").is_file():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from arxiv_collection import month_ranges, collect_month, validate_records

CATEGORIES = ["cs.AI"]
FIRST_MONTH = "2022-01"
LAST_MONTH = "2026-10"
PAGE_SIZE = 500
RUN_ID = "2026-11-12_cs-ai"  # Keep to resume; change for a new collection.
OUTPUT_DIR = project_root / "data" / "raw" / "arxiv_cs_ai_2022_2026"

windows = month_ranges(FIRST_MONTH, LAST_MONTH)
print("Category:", CATEGORIES)
print("Months:", len(windows), "from", FIRST_MONTH, "through", LAST_MONTH)
print("Output directory:", OUTPUT_DIR)


## Record this collection run

Keep `RUN_ID` when resuming this collection; use a new label and a separate output folder for a fresh collection. For example, a later information retrieval collection could use `2026-11-13_cs-ir`.

This cell creates the seven-field JSON record in `collection_runs/`. An existing record must match the category and submission dates and is preserved. A new record starts with `null` finish time and total. The completion cell below fills these only after saved-file validation succeeds. Do not edit raw data while collecting.


In [ ]:
collection_validated = False
run_path = project_root / "collection_runs" / f"{RUN_ID}.json"
if Path(RUN_ID).name != RUN_ID or not RUN_ID or RUN_ID in (".", ".."):
    raise ValueError("RUN_ID must be a simple filename label.")

run_scope = {
    "run_id": RUN_ID,
    "categories": CATEGORIES,
    "submission_start": datetime.strptime(windows[0][0], "%Y%m%d%H%M").strftime("%Y-%m-%dT%H:%M:00Z"),
    "submission_end": datetime.strptime(windows[-1][1], "%Y%m%d%H%M").strftime("%Y-%m-%dT%H:%M:59Z"),
}
if run_path.exists():
    run_record = json.loads(run_path.read_text(encoding="utf-8"))
    if any(run_record.get(key) != value for key, value in run_scope.items()):
        raise ValueError("This RUN_ID has different collection settings. Choose a new RUN_ID.")
    print("Keeping existing run record:", run_path)
else:
    run_record = {
        **run_scope,
        "retrieval_started": datetime.now(timezone.utc).isoformat(),
        "retrieval_finished": None,
        "total_records": None,
    }
    run_path.parent.mkdir(parents=True, exist_ok=True)
    with run_path.open("x", encoding="utf-8") as destination:
        destination.write(json.dumps(run_record, indent=2) + "\n")
    print("Created run record:", run_path)


## Collect and save each month

Running the next cell starts collection of all configured months. For a one-month trial, change `LAST_MONTH` to `FIRST_MONTH` above before running it.

Each request retrieves at most 500 records, with a three-second pause before requests. Completed months are reused only if settings and file hashes match. If interrupted, rerun this cell: completed months stay saved, and the unfinished month starts again. Failures stop explicitly; no automatic retries or skipped months.

Each completed window has:

- `.jsonl`: original parsed metadata, one paper per line, without text cleaning.
- `.manifest.json`: counts, query URLs, retrieval times, checks, and hashes.

Blank text is retained and flagged. Collection-blocking problems such as duplicate IDs or out-of-range dates prevent a completed checkpoint. Keep these raw files unchanged during later cleaning.


In [ ]:
collection_validated = False
manifests = []
for start_date, end_date in windows:
    manifest = collect_month(
        CATEGORIES, start_date, end_date, OUTPUT_DIR, page_size=PAGE_SIZE
    )
    manifests.append(manifest)

print("Completed or verified saved months:", len(manifests))
print("Total records saved:", sum(item["retrieved_count"] for item in manifests))


## Validate the saved collection

Re-read the monthly files, compare their counts with the API totals recorded during collection, recheck dates and category membership, and check duplicate paper IDs across months. No records are changed or removed. Any failure needs investigation before cleaning.

These checks compare against API-reported totals; they cannot prove coverage of all AI research or restore earlier text versions.


In [ ]:
collection_validated = False
validated_manifests = []
summary_rows = []
seen_ids = set()
cross_month_duplicates = 0
preview_records = []

for start_date, end_date in windows:
    stem = f"{start_date}_{end_date}"
    manifest = json.loads((OUTPUT_DIR / f"{stem}.manifest.json").read_text(encoding="utf-8"))
    expected_settings = {"categories": CATEGORIES, "start_date": start_date, "end_date": end_date}
    if manifest["status"] != "complete" or manifest["settings"] != expected_settings:
        raise RuntimeError(f"Unexpected checkpoint settings for {stem}.")
    validated_manifests.append(manifest)
    data_path = OUTPUT_DIR / manifest["data_file"]
    with data_path.open(encoding="utf-8") as source:
        records = [json.loads(line) for line in source]
    checks = validate_records(records, CATEGORIES, start_date, end_date)
    if len(records) != manifest["api_total"] or len(records) != manifest["retrieved_count"]:
        raise RuntimeError(f"Saved count does not match the checkpoint for {stem}.")
    for record in records:
        base_id = re.sub(r"v\d+$", "", record["arxiv_id"])
        if base_id in seen_ids:
            cross_month_duplicates += 1
        seen_ids.add(base_id)
    preview_records.extend(records[:max(0, 5 - len(preview_records))])
    summary_rows.append({"month": start_date[:6], "api_total": manifest["api_total"], **checks})

summary = pd.DataFrame(summary_rows)
display(summary)
print("Saved records:", int(summary["records"].sum()))
print("Unique paper IDs:", len(seen_ids))
print("Duplicate occurrences across the saved files:", cross_month_duplicates)
problem_columns = ["missing_ids", "duplicate_ids", "invalid_submission_dates",
                   "outside_date_range", "without_requested_category"]
if summary[problem_columns].to_numpy().any() or cross_month_duplicates:
    raise RuntimeError("Saved collection checks failed. Review the table before proceeding.")
print("Collection checks passed. Review blank text counts before cleaning.")
print("Blank titles:", int(summary["blank_titles"].sum()))
print("Blank abstracts:", int(summary["blank_abstracts"].sum()))
print("Earlier full-period API count for reference: 168,572 (not a fixed expected value).")

collection_validated = True


## Complete the run record

Run this cell after the validation cell passes. Times come from the monthly manifests, so reusing saved months does not pretend they were downloaded again today. The total comes from the saved-file checks. A completed record cannot be replaced with different results under the same run ID.


In [ ]:
if not collection_validated:
    raise RuntimeError("Run the saved-collection validation cell successfully first.")
if len(validated_manifests) != len(windows):
    raise RuntimeError("Not all configured months were validated.")

completed_record = {
    **run_scope,
    "retrieval_started": min(item["started_at"] for item in validated_manifests),
    "retrieval_finished": max(item["completed_at"] for item in validated_manifests),
    "total_records": int(summary["records"].sum()),
}
existing_record = json.loads(run_path.read_text(encoding="utf-8"))
if existing_record["retrieval_finished"] is not None and existing_record != completed_record:
    raise RuntimeError("The completed record differs. Review it or use a new RUN_ID.")

temporary_path = run_path.with_suffix(".json.part")
temporary_path.write_text(json.dumps(completed_record, indent=2) + "\n", encoding="utf-8")
temporary_path.replace(run_path)
print("Completed run record:", run_path)
print(json.dumps(completed_record, indent=2))


In [ ]:
# Preview only; do not clean or overwrite the monthly files here.
with pd.option_context("display.max_colwidth", 150):
    display(pd.DataFrame(preview_records))


## Review point

Share the test results and collection validation summary before moving to cleaning and exploration. The saved monthly JSONL files are the inputs for that next step. Broader category selection and classification of methods versus applications remain separate research decisions.
